Import Library

In [1]:
# Instalasi library pihak ketiga
!pip install Sastrawi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 4.6 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Bidirectional, Dense, Dropout
from tensorflow.keras.utils import to_categorical
from tqdm.notebook import tqdm # Tqdm khusus untuk notebook

# Download stopwords NLTK
nltk.download('stopwords', quiet=True)
tqdm.pandas()

Load Data Scraping & Pelabelan

In [4]:
# Load data
df = pd.read_csv('dataset_com.gojek.app_15k.csv')
df = df.dropna(subset=['content'])

# Pelabelan
def label_sentiment(score):
    if score <= 2: return 'Negatif'
    elif score == 3: return 'Netral'
    else: return 'Positif'

df['sentiment'] = df['score'].apply(label_sentiment)

print(f"Total Data: {len(df)} baris")
print("Distribusi Sentimen:\n", df['sentiment'].value_counts())

Total Data: 15000 baris
Distribusi Sentimen:
 sentiment
Positif    8948
Negatif    5403
Netral      649
Name: count, dtype: int64


Text Preprocessing

In [5]:
factory = StemmerFactory()
stemmer = factory.create_stemmer()
# Normalisasi
norm_dict = {
    # Singkatan umum
    "yg": "yang", "tdk": "tidak", "gk": "tidak", "gak": "tidak",
    "ga": "tidak", "nggak": "tidak", "bgt": "banget", "bgtt": "banget",
    "bgs": "bagus", "jlek": "jelek", "jelek": "buruk",
    "ok": "baik", "oke": "baik",

    # Masalah teknis
    "lag": "lambat", "lemot": "lambat", "ngelag": "lambat",
    "ngebug": "galat", "bug": "galat", "error": "galat",
    "crash": "gagal", "force close": "gagal",
    "loading lama": "lambat", "loading": "muat",

    # Konteks aplikasi Gojek
    "driver": "pengemudi", "drivernya": "pengemudi",
    "gojeknya": "gojek", "app": "aplikasi", "apk": "aplikasi",
    "order": "pesan", "pesen": "pesan",
    "cancel": "batal", "cancelled": "batal",
    "cust": "pelanggan", "cs": "layanan pelanggan",

    # Pembayaran
    "gopay": "gopay",  # tetap (brand)
    "saldo": "saldo",
    "topup": "isi saldo",
    "tf": "transfer",

    # Emosi / opini
    "mantap": "bagus", "keren": "bagus", "parah": "buruk",
    "kecewa": "kecewa", "puas": "puas",
    "ribet": "sulit", "susah": "sulit",
    "cepet": "cepat", "lambat banget": "sangat lambat",

    # Lain-lain
    "udh": "sudah", "sdh": "sudah", "blm": "belum",
    "trs": "terus", "krn": "karena",
    "dgn": "dengan", "sm": "sama",
    "aja": "saja"
}

def normalize_text(text):
    return ' '.join([norm_dict.get(w, w) for w in text.split()])

# Setup stopwords dan stemmer
stop_words = set(stopwords.words('indonesian'))

stop_words.update([
    # Kata umum (tidak berpengaruh ke sentimen)
    'di', 'ke', 'dari', 'yang', 'pada', 'dan', 'ini', 'itu', 'nya',
    'untuk', 'dengan', 'dalam', 'sebagai', 'oleh', 'karena',

    # Kata ganti
    'saya', 'aku', 'gue', 'gw', 'kami', 'kita',
    'kamu', 'lu', 'lo', 'anda', 'dia', 'mereka',

    # Kata tidak penting dalam opini
    'ada', 'jadi', 'aja', 'saja', 'hanya', 'cuma',
    'masih', 'sudah', 'udah', 'sdh', 'belum',
    'bisa', 'dapat', 'dapet', 'harus', 'perlu',

    # Kata penghubung / filler
    'dan', 'atau', 'tapi', 'tp', 'namun',
    'karena', 'soalnya', 'biar', 'agar',

    # Kata waktu
    'sekarang', 'dulu', 'lagi', 'terus', 'trs',
    'hari', 'minggu', 'bulan',

    # Kata umum di review (tidak terlalu penting)
    'aplikasi', 'app', 'apk',
    'gojek',
    'pengemudi', 'pelanggan',

    # Kata netral tambahan
    'menu', 'fitur', 'pakai', 'gunakan', 'dipakai',
    'buka', 'masuk', 'login'
])

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'\@\w+|\#', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    text = normalize_text(text)
    words = text.split()
    return ' '.join([stemmer.stem(w) for w in words if w not in stop_words])

print("Memulai Preprocessing... (Tunggu hingga selesai)")
df['teks_bersih'] = df['content'].progress_apply(clean_text)

# Menghapus baris yang kosong seteleh pebersihan data
df = df[df['teks_bersih'].str.strip() != '']
df = df.dropna(subset=['teks_bersih'])

print("Preprocessing Selesai!")

# Menampilkan distribusi label
print("\nDistribusi Sentimen Dataset:")
print(df['sentiment'].value_counts())

Memulai Preprocessing... (Tunggu hingga selesai)


  0%|          | 0/15000 [00:00<?, ?it/s]

Preprocessing Selesai!

Distribusi Sentimen Dataset:
sentiment
Positif    7911
Negatif    5345
Netral      636
Name: count, dtype: int64


Ekstraksi Fitur dan Encoding Label

In [6]:
# Encoding label (Y)
# Merubah teks kelas sentimen menjadi angka
encoder = LabelEncoder()
label_encoded = encoder.fit_transform(df['sentiment'].values)

# Merubah ke format One-Hot Encoding
Y = to_categorical(label_encoded)
print("Mapping Label:", dict(zip(encoder.classes_, encoder.transform(encoder.classes_))))

# Ekstraksi fitur (X)
teks = df['teks_bersih'].astype(str).values

# Parameter tokenizer
VOCAB_SIZE = 5000     # Model hanya menghafal 5000 kata unik yang sering muncul
MAX_LENGTH = 100      # Maksimal panjang kata -> 100 dimensi
TRUNC_TYPE = 'post'   # Memotong sisa kata di belakang jikalebih dari 100
PADDING_TYPE = 'post' # Menambahkan 0 di belakang kata jika kurang dari 100
OOV_TOK = "<OOV>"     # Token untuk kata yang tidak ada di 5000 kata terbanyak

# Inisialisasi dan pembuatan kamus
tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token=OOV_TOK)
tokenizer.fit_on_texts(teks)

# Proses mengubah teks ke angka (Tokenisasi)
sekuens_teks = tokenizer.texts_to_sequences(teks)

# Menyeragamkan ukuran matriks (Padding)
X = pad_sequences(sekuens_teks, maxlen=MAX_LENGTH, padding=PADDING_TYPE, truncating=TRUNC_TYPE)

print("\nBerhasil melakukan Ekstraksi Fitur!")
print("Bentuk Data X (Fitur Teks):", X.shape)
print("Bentuk Data Y (Label Sentimen):", Y.shape)

Mapping Label: {'Negatif': np.int64(0), 'Netral': np.int64(1), 'Positif': np.int64(2)}

Berhasil melakukan Ekstraksi Fitur!
Bentuk Data X (Fitur Teks): (13892, 100)
Bentuk Data Y (Label Sentimen): (13892, 3)


Pembagian Data (Train-Test Split)

In [7]:
from sklearn.model_selection import train_test_split

print("Proses pembagian data berlangsung...")

# Pembagian untuk skema 1 dan 2 (80% training, 20% testing)
X_train_80, X_test_80, y_train_80, y_test_80 = train_test_split(
    X, Y, test_size=0.2, random_state=42, stratify=Y
)

# Pembagian untuk skema 3 (90% training, 10% testing)
X_train_90, X_test_90, y_train_90, y_test_90 = train_test_split(
    X, Y, test_size=0.1, random_state=42, stratify=Y
)

print("Data Split 80/20 selesai.\nDimensi Train:", X_train_80.shape)
print("Data Split 90/10 selesai!\nDimensi Train:", X_train_90.shape)

Proses pembagian data berlangsung...
Data Split 80/20 selesai.
Dimensi Train: (11113, 100)
Data Split 90/10 selesai!
Dimensi Train: (12502, 100)


Skema 1 (Algoritma LSTM - Split 80/20)

In [8]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional, GlobalMaxPooling1D
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.regularizers import l2

print("SKEMA 1 (IMPROVED): BiLSTM + Pooling")

model_1 = Sequential([
    Embedding(input_dim=VOCAB_SIZE, output_dim=128),

    Bidirectional(LSTM(128, return_sequences=True, recurrent_dropout=0.2)),
    Dropout(0.3),

    Bidirectional(LSTM(64, return_sequences=True)),
    Dropout(0.2),

    GlobalMaxPooling1D(),

    Dense(128, activation='relu', kernel_regularizer=l2(0.001)),
    Dropout(0.3),

    Dense(3, activation='softmax')
])

optimizer = Adam(learning_rate=0.0005)

model_1.compile(
    loss='categorical_crossentropy',
    optimizer=optimizer,
    metrics=['accuracy']
)

stop_early = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)


lr_scheduler = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=2,
    min_lr=1e-5,
    verbose=1
)


history_1 = model_1.fit(
    X_train_80, y_train_80,
    epochs=30,
    batch_size=32,
    validation_data=(X_test_80, y_test_80),
    callbacks=[stop_early],
    verbose=1
)

_, acc_train_1 = model_1.evaluate(X_train_80, y_train_80, verbose=0)
_, acc_test_1 = model_1.evaluate(X_test_80, y_test_80, verbose=0)

print(f"\nHASIL SKEMA 1:\nAkurasi Training: {acc_train_1*100:.2f}%\nAkurasi Testing: {acc_test_1*100:.2f}%")

SKEMA 1 (IMPROVED): BiLSTM + Pooling
Epoch 1/30
348/348 ━━━━━━━━━━━━━━━━━━━━ 255s 707ms/step - accuracy: 0.8011 - loss: 0.6065 - val_accuracy: 0.8640 - val_loss: 0.4833
Epoch 2/30
348/348 ━━━━━━━━━━━━━━━━━━━━ 245s 705ms/step - accuracy: 0.8764 - loss: 0.4305 - val_accuracy: 0.8647 - val_loss: 0.4496
Epoch 3/30
348/348 ━━━━━━━━━━━━━━━━━━━━ 237s 683ms/step - accuracy: 0.8907 - loss: 0.3883 - val_accuracy: 0.8669 - val_loss: 0.4297
Epoch 4/30
348/348 ━━━━━━━━━━━━━━━━━━━━ 254s 730ms/step - accuracy: 0.8962 - loss: 0.3653 - val_accuracy: 0.8604 - val_loss: 0.4483
Epoch 5/30
348/348 ━━━━━━━━━━━━━━━━━━━━ 244s 702ms/step - accuracy: 0.9000 - loss: 0.3456 - val_accuracy: 0.8611 - val_loss: 0.4406
Epoch 6/30
348/348 ━━━━━━━━━━━━━━━━━━━━ 254s 679ms/step - accuracy: 0.9049 - loss: 0.3283 - val_accuracy: 0.8597 - val_loss: 0.4443

HASIL SKEMA 1:
Akurasi Training: 90.30%
Akurasi Testing: 86.69%


Skema 2 (Algoritma Bi-LSTM - Split data 80/20)
---



In [9]:
from tensorflow.keras.optimizers import Adam
print("SKEMA 2: Bi-LSTM - Split data 80/20")

model_2 = Sequential([
    Embedding(input_dim=VOCAB_SIZE, output_dim=128, input_length=MAX_LENGTH),
    Bidirectional(LSTM(64, return_sequences=True, recurrent_dropout=0.2)), # Variasi algoritma
    GlobalMaxPooling1D(),

    Dropout(0.5),
    Dense(64, activation='relu'),
    Dropout(0.4),
    Dense(3, activation='softmax')
])


custom_optimizer = Adam(learning_rate=0.0003)
model_2.compile(loss='categorical_crossentropy', optimizer=custom_optimizer, metrics=['accuracy'])

history_2 = model_2.fit(
    X_train_80, y_train_80,
    epochs=20,
    batch_size=16,
    validation_data=(X_test_80, y_test_80),
    callbacks=[stop_early],
    verbose=1
)

_, acc_train_2 = model_2.evaluate(X_train_80, y_train_80, verbose=0)
_, acc_test_2 = model_2.evaluate(X_test_80, y_test_80, verbose=0)
print(f"\nHASIL SKEMA 2\nAkurasi Training: {acc_train_2*100:.2f}%\nAkurasi Testing: {acc_test_2*100:.2f}%")

SKEMA 2: Bi-LSTM - Split data 80/20
Epoch 1/20


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


695/695 ━━━━━━━━━━━━━━━━━━━━ 141s 196ms/step - accuracy: 0.7601 - loss: 0.6065 - val_accuracy: 0.8535 - val_loss: 0.4171
Epoch 2/20
695/695 ━━━━━━━━━━━━━━━━━━━━ 141s 195ms/step - accuracy: 0.8708 - loss: 0.4054 - val_accuracy: 0.8665 - val_loss: 0.3961
Epoch 3/20
695/695 ━━━━━━━━━━━━━━━━━━━━ 142s 195ms/step - accuracy: 0.8883 - loss: 0.3586 - val_accuracy: 0.8690 - val_loss: 0.3950
Epoch 4/20
695/695 ━━━━━━━━━━━━━━━━━━━━ 149s 205ms/step - accuracy: 0.8945 - loss: 0.3283 - val_accuracy: 0.8694 - val_loss: 0.4084
Epoch 5/20
695/695 ━━━━━━━━━━━━━━━━━━━━ 138s 198ms/step - accuracy: 0.9024 - loss: 0.2991 - val_accuracy: 0.8690 - val_loss: 0.4347
Epoch 6/20
695/695 ━━━━━━━━━━━━━━━━━━━━ 133s 192ms/step - accuracy: 0.9068 - loss: 0.2749 - val_accuracy: 0.8658 - val_loss: 0.4615

HASIL SKEMA 2
Akurasi Training: 90.25%
Akurasi Testing: 86.90%


Skema 3 (Algoritma Bi-LSTM - Split data 90/10)

In [10]:
print("SKEMA 3: Bi-LSTM - Split data 90/10")

model_3 = Sequential([
    Embedding(input_dim=VOCAB_SIZE, output_dim=128),
    Bidirectional(LSTM(128, return_sequences=True, recurrent_dropout=0.2)),
    GlobalMaxPooling1D(),

    Dense(128, activation='relu'),
    Dropout(0.5),

    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(3, activation='softmax')
])

optimizer = Adam(learning_rate=0.0003)

model_3.compile(loss='categorical_crossentropy', optimizer=optimizer, metrics=['accuracy'])

history_3 = model_3.fit(
    X_train_90, y_train_90,
    epochs=20,
    batch_size=16,
    validation_data=(X_test_90, y_test_90),
    callbacks=[stop_early],
    verbose=1
)

_, acc_train_3 = model_3.evaluate(X_train_90, y_train_90, verbose=0)
_, acc_test_3 = model_3.evaluate(X_test_90, y_test_90, verbose=0)
print(f"\nHASIL SKEMA\nAkurasi Training: {acc_train_3*100:.2f}%\nAkurasi Testing: {acc_test_3*100:.2f}%")

SKEMA 3: Bi-LSTM - Split data 90/10
Epoch 1/20
782/782 ━━━━━━━━━━━━━━━━━━━━ 324s 407ms/step - accuracy: 0.7940 - loss: 0.5406 - val_accuracy: 0.8662 - val_loss: 0.4174
Epoch 2/20
782/782 ━━━━━━━━━━━━━━━━━━━━ 313s 400ms/step - accuracy: 0.8799 - loss: 0.3809 - val_accuracy: 0.8669 - val_loss: 0.4158
Epoch 3/20
782/782 ━━━━━━━━━━━━━━━━━━━━ 328s 408ms/step - accuracy: 0.8943 - loss: 0.3341 - val_accuracy: 0.8669 - val_loss: 0.4231

HASIL SKEMA
Akurasi Training: 88.25%
Akurasi Testing: 86.62%


In [11]:
!pip freeze > requirements.txt

In [12]:
from google.colab import files
files.download('requirements.txt')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [13]:
import numpy as np

print("======================================================")
print("  TAHAP INFERENSI: PENGUJIAN MODEL DENGAN DATA BARU   ")
print("======================================================\n")

# Data dumy sebagai contoh ulasan baru (Mewakili Positif, Negatif, dan Netral)
data_uji_baru = [
    "Aplikasinya bagus banget, driver cepat datang dan pelayanannya memuaskan",
    "Gojek sangat membantu, mantap lah",
    "Mantap sih, pesan makanan jadi lebih cepat dan gak ribet",
    "Aplikasi sering error dan susah login, sangat mengecewakan",
    "Driver sering cancel tanpa alasan",
    "Lemot banget aplikasinya, tiap buka pasti loading lama",
    "Aplikasi cukup membantu untuk aktivitas sehari hari",
]

def prediksi_sentimen(teks_baru):
    # Preprocessing -> Membersihkan teks menggunakan fungsi clean_text
    teks_bersih = clean_text(teks_baru)

    # Ekstraksi Fitur -> Menggunakan tokenizer dan padding dari data training
    sekuens = tokenizer.texts_to_sequences([teks_bersih])
    pad_sekuens = pad_sequences(sekuens, maxlen=MAX_LENGTH, padding='post', truncating='post')

    # Prediksi -> Menggunakan model terbaik
    prediksi_prob = model_2.predict(pad_sekuens, verbose=0)
    kelas_prediksi = np.argmax(prediksi_prob, axis=1)[0]

    # Decode Label: Mengubah angka kembali menjadi teks kategori
    label_kategori = encoder.inverse_transform([kelas_prediksi])[0]

    return label_kategori.upper()

# Eksekusi prediksi untuk setiap kalimat uji
for i, teks in enumerate(data_uji_baru, 1):
    hasil = prediksi_sentimen(teks)
    print(f"Data Uji Ke-{i}")
    print(f"Ulasan Asli    : '{teks}'")
    print(f"Hasil Prediksi : >>> {hasil} <<<")
    print("-" * 60)

  TAHAP INFERENSI: PENGUJIAN MODEL DENGAN DATA BARU   

Data Uji Ke-1
Ulasan Asli    : 'Aplikasinya bagus banget, driver cepat datang dan pelayanannya memuaskan'
Hasil Prediksi : >>> POSITIF <<<
------------------------------------------------------------
Data Uji Ke-2
Ulasan Asli    : 'Gojek sangat membantu, mantap lah'
Hasil Prediksi : >>> POSITIF <<<
------------------------------------------------------------
Data Uji Ke-3
Ulasan Asli    : 'Mantap sih, pesan makanan jadi lebih cepat dan gak ribet'
Hasil Prediksi : >>> POSITIF <<<
------------------------------------------------------------
Data Uji Ke-4
Ulasan Asli    : 'Aplikasi sering error dan susah login, sangat mengecewakan'
Hasil Prediksi : >>> NEGATIF <<<
------------------------------------------------------------
Data Uji Ke-5
Ulasan Asli    : 'Driver sering cancel tanpa alasan'
Hasil Prediksi : >>> NEGATIF <<<
------------------------------------------------------------
Data Uji Ke-6
Ulasan Asli    : 'Lemot banget aplikas